> **Versión GitHub reproducible.** Esta copia no modifica el notebook original de Google Drive. Las rutas personales de Drive se han sustituido por rutas del repositorio. Los datos brutos de gran tamaño no incluidos en GitHub deben descargarse según `docs/DATA_SOURCES.md`.


In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS - VERSIÓN GITHUB / GOOGLE COLAB
# ============================================================
from pathlib import Path
import os
import subprocess

ROOT = Path("/content/TFM_Rendimiento_cultivos")

# Si el notebook se abre directamente desde GitHub en Colab,
# descargamos automáticamente el repositorio.
if not ROOT.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ritaridh/tfm-crop-yield-prediction.git",
            str(ROOT),
        ],
        check=True,
    )

os.chdir(ROOT)

RUTA_TFM = str(ROOT)
RUTA_BASE = str(ROOT)
BASE = str(ROOT)
BASE_DIR = ROOT
RUTA_ORIGINALES = str(ROOT / "01_Datos_originales")
RUTA_EUROCROPS = str(ROOT / "01_Datos_originales" / "EuroCrops")
RUTA_MAPA = str(ROOT / "01_Datos_originales" / "MAPA")
RUTA_SENTINEL = str(ROOT / "01_Datos_originales" / "Sentinel2")
RUTA_PROCESADOS = str(ROOT / "02_Datos_procesados")
RUTA_RESULTADOS = str(ROOT / "05_Resultados")
PROC = RUTA_PROCESADOS
RESULTADOS = RUTA_RESULTADOS

Path(RUTA_PROCESADOS).mkdir(parents=True, exist_ok=True)
Path(RUTA_RESULTADOS).mkdir(parents=True, exist_ok=True)

print("Repositorio:", ROOT)


In [ ]:
# ============================================================
# PROPIEDADES EDÁFICAS
# Predicción de rendimiento de trigo y cebada
# ============================================================


import os
import pandas as pd
import numpy as np
import geopandas as gpd

# ------------------------------------------------------------
# RUTAS PERMANENTES EN GOOGLE DRIVE
# ------------------------------------------------------------

BASE = "/content/TFM_Rendimiento_cultivos"

RUTA_ORIGINALES = os.path.join(
    BASE,
    "01_Datos_originales"
)

RUTA_PROCESADOS = os.path.join(
    BASE,
    "02_Datos_procesados"
)

RUTA_RESULTADOS = os.path.join(
    BASE,
    "05_Resultados"
)

print("BASE:", BASE)
print("Originales:", RUTA_ORIGINALES)
print("Procesados:", RUTA_PROCESADOS)
print("Resultados:", RUTA_RESULTADOS)

In [ ]:
# ============================================================
# CARGAR DATASET MAPA + ERA5 YA PROCESADO
# ============================================================

ruta_dataset = os.path.join(
    RUTA_PROCESADOS,
    "dataset_MAPA_ERA5_trigo_cebada_2017_2023.csv"
)

dataset = pd.read_csv(ruta_dataset)

print("Dataset MAPA + ERA5")
print("Dimensiones:", dataset.shape)

print("\nCultivos:")
print(dataset["cultivo"].value_counts())

print("\nAños:")
print(sorted(dataset["ano"].unique()))

print("\nNulos totales:")
print(dataset.isna().sum().sum())

display(dataset.head())

In [ ]:
# ============================================================
# CARGAR GEOMETRÍAS PROVINCIALES
# ============================================================

ruta_provincias = os.path.join(
    RUTA_PROCESADOS,
    "provincias_CNIG_50.gpkg"
)

provincias = gpd.read_file(ruta_provincias)

print("Provincias:", len(provincias))
print("CRS:", provincias.crs)

display(provincias.head())

In [ ]:
# ============================================================
# INVENTARIO DE ARCHIVOS DE SUELO / RASTER
# ============================================================

extensiones_raster = (
    ".tif",
    ".tiff",
    ".img",
    ".nc"
)

rasters_encontrados = []

for raiz, carpetas, archivos in os.walk(RUTA_ORIGINALES):

    for archivo in archivos:

        if archivo.lower().endswith(extensiones_raster):

            rasters_encontrados.append(
                os.path.join(raiz, archivo)
            )

print(
    "Número de archivos raster encontrados:",
    len(rasters_encontrados)
)

for ruta in rasters_encontrados:
    print(ruta)

# SoilGrids

In [ ]:
# ============================================================
# CREAR CARPETA SOILGRIDS EN DRIVE
# ============================================================

import os

RUTA_SOILGRIDS = os.path.join(
    RUTA_ORIGINALES,
    "SoilGrids"
)

os.makedirs(RUTA_SOILGRIDS, exist_ok=True)

print("Ruta SoilGrids:")
print(RUTA_SOILGRIDS)

print("Existe:", os.path.exists(RUTA_SOILGRIDS))

In [ ]:
!pip -q install owslib rasterio

In [ ]:
from owslib.wcs import WebCoverageService
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
import os

print("Librerías cargadas correctamente.")

In [ ]:
#------------------------------------------------
# Preparar extensión de España
#------------------------------------------------

ruta_provincias = os.path.join(
    RUTA_PROCESADOS,
    "provincias_CNIG_50.gpkg"
)

provincias = gpd.read_file(ruta_provincias)

print("Provincias:", len(provincias))
print("CRS:", provincias.crs)

minx, miny, maxx, maxy = provincias.total_bounds

print("\nBounding box España:")
print("Oeste:", minx)
print("Sur:", miny)
print("Este:", maxx)
print("Norte:", maxy)

In [ ]:
#------------------------------------------
# Comprobar el servicio SoilGrids antes de descargar
#------------------------------------------

from owslib.wcs import WebCoverageService

url_wcs = "https://maps.isric.org/mapserv?map=/map/clay.map"

wcs = WebCoverageService(
    url_wcs,
    version="2.0.1"
)

print("Servicio conectado.")
print("\nCoberturas disponibles:")

for cobertura in list(wcs.contents.keys())[:20]:
    print(cobertura)

In [ ]:
# ============================================================
# INSPECCIONAR LA COBERTURA SOILGRIDS
# ============================================================

cobertura = "clay_0-5cm_mean"

info = wcs.contents[cobertura]

print("Cobertura:", cobertura)

print("\nBounding boxes:")
print(info.boundingboxes)

print("\nCRS soportados:")
print(info.supportedCRS)

print("\nFormatos soportados:")
print(info.supportedFormats)

print("\nGrid:")
print(info.grid)

In [ ]:
# ============================================================
# INSPECCIONAR LA COBERTURA SOILGRIDS
# ============================================================

cobertura = "clay_0-5cm_mean"

info = wcs.contents[cobertura]

print("Cobertura:", cobertura)

print("\nBounding boxes:")
print(info.boundingboxes)

print("\nCRS soportados:")
print(info.supportedCRS)

print("\nFormatos soportados:")
print(info.supportedFormats)

print("\nGrid:")
print(info.grid)

In [ ]:
# ============================================================
# PASO 9B
# Bounding box de España en WGS84
# ============================================================

# Asegurar que las provincias están en WGS84
provincias_wgs84 = provincias.to_crs("EPSG:4326")

minx, miny, maxx, maxy = provincias_wgs84.total_bounds

print("CRS provincias:", provincias_wgs84.crs)

print("\nBounding box España:")
print("Longitud mínima:", minx)
print("Latitud mínima:", miny)
print("Longitud máxima:", maxx)
print("Latitud máxima:", maxy)

In [ ]:
# ============================================================
# PASO 9C - DESCARGA DE PRUEBA
# SoilGrids clay 0-5 cm
# ============================================================

import requests
import os

cobertura = "clay_0-5cm_mean"

ruta_clay_prueba = os.path.join(
    RUTA_SOILGRIDS,
    "clay_0-5cm_mean_spain.tif"
)

url = "https://maps.isric.org/mapserv"

params = {
    "map": "/map/clay.map",
    "SERVICE": "WCS",
    "VERSION": "2.0.1",
    "REQUEST": "GetCoverage",
    "COVERAGEID": cobertura,
    "FORMAT": "GEOTIFF_INT16",

    # Bounding box interpretado en WGS84
    "SUBSET": [
        f"Long({minx},{maxx})",
        f"Lat({miny},{maxy})"
    ],

    "SUBSETTINGCRS":
        "http://www.opengis.net/def/crs/EPSG/0/4326",

    "OUTPUTCRS":
        "http://www.opengis.net/def/crs/EPSG/0/4326"
}

print("Solicitando:", cobertura)
print("Destino:", ruta_clay_prueba)

respuesta = requests.get(
    url,
    params=params,
    timeout=300
)

print("\nHTTP status:", respuesta.status_code)
print("Content-Type:", respuesta.headers.get("Content-Type"))
print("Tamaño recibido:", len(respuesta.content), "bytes")

In [ ]:
# ============================================================
# PASO 9D - GUARDAR EL GEOTIFF EN GOOGLE DRIVE
# ============================================================

with open(ruta_clay_prueba, "wb") as f:
    f.write(respuesta.content)

print("Archivo guardado:")
print(ruta_clay_prueba)

print("\nExiste:", os.path.exists(ruta_clay_prueba))

print(
    "Tamaño MB:",
    round(
        os.path.getsize(ruta_clay_prueba) / (1024**2),
        2
    )
)

In [ ]:
# ============================================================
# PASO 10 - VALIDAR SOILGRIDS clay 0-5 cm
# ============================================================

import rasterio
import numpy as np

with rasterio.open(ruta_clay_prueba) as src:

    print("CRS:", src.crs)
    print("Resolución:", src.res)
    print("Dimensiones:", src.width, "x", src.height)
    print("Número de bandas:", src.count)
    print("NoData:", src.nodata)
    print("Extensión:", src.bounds)
    print("Tipo de datos:", src.dtypes)

    datos = src.read(1)

    # Seleccionar valores válidos
    if src.nodata is not None:
        validos = datos[
            (datos != src.nodata) &
            np.isfinite(datos)
        ]
    else:
        validos = datos[
            np.isfinite(datos)
        ]

    print("\nNúmero de píxeles válidos:")
    print(len(validos))

    print("\nValores SoilGrids originales:")
    print("Mínimo:", float(np.min(validos)))
    print("Máximo:", float(np.max(validos)))
    print("Media:", float(np.mean(validos)))
    print("Mediana:", float(np.median(validos)))

    # clay SoilGrids: g/kg
    # conversión a porcentaje: dividir entre 10
    clay_pct = validos / 10.0

    print("\nArcilla (%):")
    print("Mínimo:", float(np.min(clay_pct)))
    print("Máximo:", float(np.max(clay_pct)))
    print("Media:", float(np.mean(clay_pct)))
    print("Mediana:", float(np.median(clay_pct)))

In [ ]:
# ============================================================
# PASO 11 - ANALIZAR SOILGRIDS SOLO DENTRO DE ESPAÑA
# ============================================================

import rasterio
from rasterio.mask import mask
import numpy as np

# Las provincias ya están en EPSG:4326,
# igual que el raster SoilGrids descargado
print("CRS provincias:", provincias.crs)

with rasterio.open(ruta_clay_prueba) as src:

    print("CRS raster:", src.crs)

    # Geometrías de las 50 provincias
    geometrias = [
        geom.__geo_interface__
        for geom in provincias.geometry
    ]

    # Recortar usando los polígonos provinciales
    datos_recortados, transform_recorte = mask(
        src,
        geometrias,
        crop=True,
        filled=False
    )

    # Primera banda
    datos_espana = datos_recortados[0]

    # Solo píxeles dentro de las geometrías
    validos_espana = datos_espana.compressed()

print("\nPíxeles dentro de las provincias:")
print(len(validos_espana))

print("\nNúmero de valores = 0:")
print(np.sum(validos_espana == 0))

print("\nPorcentaje de valores = 0:")
print(
    round(
        100 * np.mean(validos_espana == 0),
        4
    ),
    "%"
)

print("\nValores originales dentro de España:")
print("Mínimo:", float(np.min(validos_espana)))
print("Máximo:", float(np.max(validos_espana)))
print("Media:", float(np.mean(validos_espana)))
print("Mediana:", float(np.median(validos_espana)))

# Estadísticas excluyendo temporalmente los 0,
# SOLO PARA DIAGNÓSTICO
positivos = validos_espana[validos_espana > 0]

print("\nValores > 0, solo para diagnóstico:")
print("Número:", len(positivos))
print("Mínimo:", float(np.min(positivos)))
print("Máximo:", float(np.max(positivos)))
print("Media:", float(np.mean(positivos)))
print("Mediana:", float(np.median(positivos)))

print("\nArcilla (%) para valores > 0:")
print("Mínimo:", float(np.min(positivos) / 10))
print("Máximo:", float(np.max(positivos) / 10))
print("Media:", float(np.mean(positivos) / 10))
print("Mediana:", float(np.median(positivos) / 10))

In [ ]:
# ============================================================
# PASO 12 - CONFIGURAR CAPAS SOILGRIDS
# ============================================================

propiedades_soilgrids = {
    "clay": "Contenido de arcilla",
    "sand": "Contenido de arena",
    "soc": "Carbono orgánico del suelo",
    "phh2o": "pH del suelo",
    "bdod": "Densidad aparente"
}

profundidades = [
    "0-5cm",
    "5-15cm",
    "15-30cm"
]

capas_soilgrids = []

for propiedad in propiedades_soilgrids:
    for profundidad in profundidades:

        cobertura = f"{propiedad}_{profundidad}_mean"

        capas_soilgrids.append({
            "propiedad": propiedad,
            "profundidad": profundidad,
            "cobertura": cobertura
        })

print("Número de propiedades:", len(propiedades_soilgrids))
print("Número de profundidades:", len(profundidades))
print("Número total de capas:", len(capas_soilgrids))

print("\nCapas que se descargarán:")

for capa in capas_soilgrids:
    print(capa["cobertura"])

In [ ]:
# ============================================================
# PASO 13 - COMPROBAR COBERTURAS SOILGRIDS
# ============================================================

from owslib.wcs import WebCoverageService

resultado_coberturas = []

for propiedad in propiedades_soilgrids:

    print("\n" + "=" * 60)
    print("Propiedad:", propiedad)

    url_wcs = (
        f"https://maps.isric.org/mapserv?"
        f"map=/map/{propiedad}.map"
    )

    try:
        wcs_prop = WebCoverageService(
            url_wcs,
            version="2.0.1"
        )

        coberturas_disponibles = list(
            wcs_prop.contents.keys()
        )

        for profundidad in profundidades:

            cobertura = (
                f"{propiedad}_{profundidad}_mean"
            )

            existe = cobertura in coberturas_disponibles

            resultado_coberturas.append({
                "propiedad": propiedad,
                "profundidad": profundidad,
                "cobertura": cobertura,
                "existe": existe
            })

            print(
                cobertura,
                "->",
                "OK" if existe else "NO ENCONTRADA"
            )

    except Exception as e:

        print("ERROR accediendo al servicio:")
        print(e)

In [ ]:
import pandas as pd

comprobacion_soilgrids = pd.DataFrame(
    resultado_coberturas
)

display(comprobacion_soilgrids)

print(
    "\nCoberturas encontradas:",
    comprobacion_soilgrids["existe"].sum(),
    "/",
    len(comprobacion_soilgrids)
)

In [ ]:
# ============================================================
# PASO 14 - DESCARGAR LAS 15 CAPAS SOILGRIDS
# DIRECTAMENTE A GOOGLE DRIVE
# ============================================================

import os
import requests
import pandas as pd

os.makedirs(RUTA_SOILGRIDS, exist_ok=True)

# Bounding box de las 50 provincias españolas
provincias_wgs84 = provincias.to_crs("EPSG:4326")
minx, miny, maxx, maxy = provincias_wgs84.total_bounds

print("Bounding box utilizado:")
print("Longitud:", minx, "a", maxx)
print("Latitud:", miny, "a", maxy)

resultados_descarga = []

for capa in capas_soilgrids:

    propiedad = capa["propiedad"]
    profundidad = capa["profundidad"]
    cobertura = capa["cobertura"]

    archivo_salida = os.path.join(
        RUTA_SOILGRIDS,
        f"{cobertura}_spain.tif"
    )

    print("\n" + "=" * 65)
    print("Capa:", cobertura)

    # --------------------------------------------------------
    # Si ya existe, no volver a descargar
    # --------------------------------------------------------

    if os.path.exists(archivo_salida):

        tamano_mb = (
            os.path.getsize(archivo_salida) /
            (1024 ** 2)
        )

        print("Ya existe -> se omite")
        print("Tamaño:", round(tamano_mb, 2), "MB")

        resultados_descarga.append({
            "propiedad": propiedad,
            "profundidad": profundidad,
            "cobertura": cobertura,
            "estado": "YA_EXISTIA",
            "tamano_MB": round(tamano_mb, 2)
        })

        continue

    # --------------------------------------------------------
    # Servicio WCS correspondiente a la propiedad
    # --------------------------------------------------------

    url = "https://maps.isric.org/mapserv"

    params = [
        ("map", f"/map/{propiedad}.map"),
        ("SERVICE", "WCS"),
        ("VERSION", "2.0.1"),
        ("REQUEST", "GetCoverage"),
        ("COVERAGEID", cobertura),
        ("FORMAT", "GEOTIFF_INT16"),

        ("SUBSET", f"Long({minx},{maxx})"),
        ("SUBSET", f"Lat({miny},{maxy})"),

        (
            "SUBSETTINGCRS",
            "http://www.opengis.net/def/crs/EPSG/0/4326"
        ),

        (
            "OUTPUTCRS",
            "http://www.opengis.net/def/crs/EPSG/0/4326"
        )
    ]

    try:

        print("Descargando...")

        respuesta = requests.get(
            url,
            params=params,
            timeout=600
        )

        content_type = respuesta.headers.get(
            "Content-Type",
            ""
        )

        print("HTTP:", respuesta.status_code)
        print("Content-Type:", content_type)

        # ----------------------------------------------------
        # Validar antes de guardar
        # ----------------------------------------------------

        if (
            respuesta.status_code == 200
            and "tiff" in content_type.lower()
        ):

            with open(archivo_salida, "wb") as f:
                f.write(respuesta.content)

            tamano_mb = (
                os.path.getsize(archivo_salida) /
                (1024 ** 2)
            )

            print("OK")
            print("Guardado en:")
            print(archivo_salida)
            print("Tamaño:", round(tamano_mb, 2), "MB")

            resultados_descarga.append({
                "propiedad": propiedad,
                "profundidad": profundidad,
                "cobertura": cobertura,
                "estado": "DESCARGADO",
                "tamano_MB": round(tamano_mb, 2)
            })

        else:

            print("ERROR: respuesta no válida")

            print(
                respuesta.text[:500]
                if "text" in content_type.lower()
                else "Respuesta no textual"
            )

            resultados_descarga.append({
                "propiedad": propiedad,
                "profundidad": profundidad,
                "cobertura": cobertura,
                "estado": "ERROR",
                "tamano_MB": None
            })

    except Exception as e:

        print("ERROR:")
        print(e)

        resultados_descarga.append({
            "propiedad": propiedad,
            "profundidad": profundidad,
            "cobertura": cobertura,
            "estado": "ERROR",
            "tamano_MB": None
        })


# ============================================================
# RESUMEN
# ============================================================

descargas_soilgrids_df = pd.DataFrame(
    resultados_descarga
)

print("\n" + "=" * 65)
print("RESUMEN DE DESCARGA")
print("=" * 65)

display(descargas_soilgrids_df)

print("\nEstados:")
print(
    descargas_soilgrids_df["estado"]
    .value_counts()
)

In [ ]:
# ============================================================
# PASO 15 - VALIDACIÓN DE LOS 15 RASTERS SOILGRIDS
# ============================================================

import os
import rasterio
import numpy as np
import pandas as pd

validacion_rasters = []

for capa in capas_soilgrids:

    propiedad = capa["propiedad"]
    profundidad = capa["profundidad"]
    cobertura = capa["cobertura"]

    ruta = os.path.join(
        RUTA_SOILGRIDS,
        f"{cobertura}_spain.tif"
    )

    registro = {
        "propiedad": propiedad,
        "profundidad": profundidad,
        "cobertura": cobertura,
        "existe": os.path.exists(ruta)
    }

    if not os.path.exists(ruta):
        validacion_rasters.append(registro)
        continue

    try:

        with rasterio.open(ruta) as src:

            datos = src.read(1)

            # Para esta auditoría separamos los valores > 0.
            # No estamos modificando todavía el raster.
            positivos = datos[
                np.isfinite(datos) &
                (datos > 0)
            ]

            registro.update({
                "tamano_MB": round(
                    os.path.getsize(ruta) / (1024**2),
                    2
                ),
                "crs": str(src.crs),
                "ancho": src.width,
                "alto": src.height,
                "res_x": src.res[0],
                "res_y": src.res[1],
                "dtype": src.dtypes[0],
                "nodata": src.nodata,
                "min_positivo": (
                    float(np.min(positivos))
                    if len(positivos) > 0 else np.nan
                ),
                "max_positivo": (
                    float(np.max(positivos))
                    if len(positivos) > 0 else np.nan
                ),
                "media_positivos": (
                    float(np.mean(positivos))
                    if len(positivos) > 0 else np.nan
                )
            })

    except Exception as e:

        registro["error"] = str(e)

    validacion_rasters.append(registro)


validacion_soilgrids_df = pd.DataFrame(
    validacion_rasters
)

display(validacion_soilgrids_df)

In [ ]:
# ============================================================
# PASO 15B - RESUMEN DE INTEGRIDAD
# ============================================================

print(
    "Archivos encontrados:",
    validacion_soilgrids_df["existe"].sum(),
    "/",
    len(validacion_soilgrids_df)
)

print("\nCRS encontrados:")
print(
    validacion_soilgrids_df["crs"]
    .value_counts(dropna=False)
)

print("\nDimensiones distintas:")
display(
    validacion_soilgrids_df[
        ["ancho", "alto"]
    ]
    .drop_duplicates()
)

print("\nTipos de datos:")
print(
    validacion_soilgrids_df["dtype"]
    .value_counts(dropna=False)
)

print("\nNoData:")
print(
    validacion_soilgrids_df["nodata"]
    .value_counts(dropna=False)
)

print("\nResumen de rangos:")
display(
    validacion_soilgrids_df[
        [
            "propiedad",
            "profundidad",
            "min_positivo",
            "max_positivo",
            "media_positivos"
        ]
    ]
)

# Provincias para SoilGrids

In [ ]:
# ============================================================
# PASO 17A - PREPARAR PROVINCIAS PARA SOILGRIDS
# ============================================================

import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
import pandas as pd
import os

# Usamos el fichero provincial definitivo que ya guardamos
ruta_provincias = os.path.join(
    RUTA_PROCESADOS,
    "provincias_CNIG_50.gpkg"
)

provincias_soil = gpd.read_file(
    ruta_provincias,
    layer="provincias"
)

# SoilGrids está en EPSG:4326
provincias_soil = provincias_soil.to_crs(
    "EPSG:4326"
)

print("Número de provincias:", len(provincias_soil))
print("CRS:", provincias_soil.crs)

print("\nColumnas:")
print(provincias_soil.columns.tolist())

print("\nPrimeras provincias:")
display(
    provincias_soil[
        ["provincia_estandar"]
    ].head(10)
)

In [ ]:
# ============================================================
# PASO 17B - FUNCIÓN DE ESTADÍSTICAS PROVINCIALES
# ============================================================

def estadisticas_raster_por_provincia(
    ruta_raster,
    provincias_gdf
):

    resultados = []

    with rasterio.open(ruta_raster) as src:

        for _, provincia in provincias_gdf.iterrows():

            nombre_provincia = provincia[
                "provincia_estandar"
            ]

            geometria = [
                provincia.geometry.__geo_interface__
            ]

            try:

                datos, _ = rasterio.mask.mask(
                    src,
                    geometria,
                    crop=True,
                    filled=False
                )

                valores = datos[0]

                # Extraer únicamente píxeles no enmascarados
                if np.ma.isMaskedArray(valores):
                    valores = valores.compressed()
                else:
                    valores = valores.ravel()

                # Valores numéricamente válidos
                valores = valores[
                    np.isfinite(valores)
                ]

                # Los 0 se consideran ausencia de predicción
                valores = valores[
                    valores > 0
                ]

                if len(valores) > 0:

                    resultados.append({
                        "provincia_estandar":
                            nombre_provincia,

                        "media_original":
                            float(np.mean(valores)),

                        "mediana_original":
                            float(np.median(valores)),

                        "n_pixeles_validos":
                            len(valores)
                    })

                else:

                    resultados.append({
                        "provincia_estandar":
                            nombre_provincia,

                        "media_original":
                            np.nan,

                        "mediana_original":
                            np.nan,

                        "n_pixeles_validos":
                            0
                    })

            except Exception as e:

                print(
                    "ERROR:",
                    nombre_provincia,
                    e
                )

                resultados.append({
                    "provincia_estandar":
                        nombre_provincia,

                    "media_original":
                        np.nan,

                    "mediana_original":
                        np.nan,

                    "n_pixeles_validos":
                        0
                })

    return pd.DataFrame(resultados)

In [ ]:
# ============================================================
# PASO 17C - PRUEBA CON CLAY 0-5 CM
# ============================================================

ruta_prueba = os.path.join(
    RUTA_SOILGRIDS,
    "clay_0-5cm_mean_spain.tif"
)

prueba_clay = estadisticas_raster_por_provincia(
    ruta_prueba,
    provincias_soil
)

# Conversión SoilGrids:
# clay: g/kg -> %
prueba_clay["clay_0_5cm_pct"] = (
    prueba_clay["media_original"] / 10
)

print(
    "Número de provincias:",
    len(prueba_clay)
)

print(
    "Provincias sin datos:",
    prueba_clay["clay_0_5cm_pct"]
    .isna()
    .sum()
)

display(
    prueba_clay.sort_values(
        "clay_0_5cm_pct"
    )
)

In [ ]:
# ============================================================
# PASO 18 - EXTRAER LAS 15 CAPAS SOILGRIDS POR PROVINCIA
# ============================================================

import os
import pandas as pd
import numpy as np

# Factores de conversión de los valores almacenados
factores_conversion = {
    "clay": 10,    # g/kg -> %
    "sand": 10,    # g/kg -> %
    "soc": 10,     # dg/kg -> g/kg
    "phh2o": 10,   # pH x 10 -> pH
    "bdod": 100    # cg/cm3 -> g/cm3
}

resultados_soil = []

for capa in capas_soilgrids:

    propiedad = capa["propiedad"]
    profundidad = capa["profundidad"]
    cobertura = capa["cobertura"]

    ruta = os.path.join(
        RUTA_SOILGRIDS,
        f"{cobertura}_spain.tif"
    )

    print("\nProcesando:", cobertura)

    df_tmp = estadisticas_raster_por_provincia(
        ruta,
        provincias_soil
    )

    # Convertir a las unidades que utilizaremos
    df_tmp["valor"] = (
        df_tmp["media_original"] /
        factores_conversion[propiedad]
    )

    df_tmp["propiedad"] = propiedad
    df_tmp["profundidad"] = profundidad

    resultados_soil.append(
        df_tmp[
            [
                "provincia_estandar",
                "propiedad",
                "profundidad",
                "valor",
                "n_pixeles_validos"
            ]
        ]
    )

soil_long = pd.concat(
    resultados_soil,
    ignore_index=True
)

print("\nProcesamiento terminado.")
print("Número de registros:", len(soil_long))

display(soil_long.head(20))

In [ ]:
# ============================================================
# PASO 19 - CONTROL DE CALIDAD
# ============================================================

print("Número de provincias:")
print(soil_long["provincia_estandar"].nunique())

print("\nNúmero de propiedades:")
print(soil_long["propiedad"].nunique())

print("\nPropiedades:")
print(sorted(soil_long["propiedad"].unique()))

print("\nProfundidades:")
print(sorted(soil_long["profundidad"].unique()))

print("\nValores nulos:")
print(soil_long["valor"].isna().sum())

print("\nNúmero de registros por propiedad:")
print(
    soil_long.groupby("propiedad")
    .size()
)

print("\nRangos después de conversión:")

display(
    soil_long.groupby("propiedad")["valor"]
    .agg(["min", "mean", "median", "max"])
)

In [ ]:
# ============================================================
# PASO 20 - MEDIA PONDERADA 0-30 CM
# ============================================================

pesos_profundidad = {
    "0-5cm": 5,
    "5-15cm": 10,
    "15-30cm": 15
}

soil_long["peso_cm"] = (
    soil_long["profundidad"]
    .map(pesos_profundidad)
)

soil_long["valor_ponderado"] = (
    soil_long["valor"] *
    soil_long["peso_cm"]
)

soil_0_30 = (
    soil_long
    .groupby(
        [
            "provincia_estandar",
            "propiedad"
        ],
        as_index=False
    )
    .agg(
        suma_ponderada=(
            "valor_ponderado",
            "sum"
        ),
        suma_pesos=(
            "peso_cm",
            "sum"
        )
    )
)

soil_0_30["valor_0_30"] = (
    soil_0_30["suma_ponderada"] /
    soil_0_30["suma_pesos"]
)

print(
    "Registros 0-30 cm:",
    len(soil_0_30)
)

print(
    "Provincias:",
    soil_0_30["provincia_estandar"].nunique()
)

display(soil_0_30.head(15))

In [ ]:
# ============================================================
# PASO 21 - DATASET EDÁFICO 0-30 CM EN FORMATO ANCHO
# ============================================================

soil_provincial = (
    soil_0_30
    .pivot(
        index="provincia_estandar",
        columns="propiedad",
        values="valor_0_30"
    )
    .reset_index()
)

# Nombres claros y con unidades explícitas
soil_provincial = soil_provincial.rename(
    columns={
        "clay": "clay_pct_0_30cm",
        "sand": "sand_pct_0_30cm",
        "soc": "soc_g_kg_0_30cm",
        "phh2o": "ph_0_30cm",
        "bdod": "bdod_g_cm3_0_30cm"
    }
)

print("Dimensiones:", soil_provincial.shape)

print("\nColumnas:")
print(soil_provincial.columns.tolist())

display(soil_provincial.head(10))

In [ ]:
# ============================================================
# PASO 22 - CONTROL FÍSICO DE VARIABLES EDÁFICAS
# ============================================================

soil_provincial["clay_sand_pct"] = (
    soil_provincial["clay_pct_0_30cm"] +
    soil_provincial["sand_pct_0_30cm"]
)

print("CONTROL DE RANGOS")
print("=" * 60)

variables_control = [
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm",
    "clay_sand_pct"
]

display(
    soil_provincial[variables_control]
    .describe()
    .T
)

print("\nValores nulos:")
print(
    soil_provincial[
        variables_control
    ].isna().sum()
)

print(
    "\nProvincias con arcilla + arena > 100 %:",
    (
        soil_provincial["clay_sand_pct"] > 100
    ).sum()
)

In [ ]:
# ============================================================
# PASO 23 - ESTIMAR LIMO PARA CONTROL DE CONSISTENCIA
# ============================================================

soil_provincial["silt_pct_0_30cm"] = (
    100
    - soil_provincial["clay_pct_0_30cm"]
    - soil_provincial["sand_pct_0_30cm"]
)

print("Resumen de limo estimado:")

print(
    soil_provincial[
        "silt_pct_0_30cm"
    ].describe()
)

print(
    "\nProvincias con limo < 0:",
    (
        soil_provincial[
            "silt_pct_0_30cm"
        ] < 0
    ).sum()
)

display(
    soil_provincial[
        [
            "provincia_estandar",
            "clay_pct_0_30cm",
            "sand_pct_0_30cm",
            "silt_pct_0_30cm"
        ]
    ]
    .sort_values("silt_pct_0_30cm")
)

# Guardar datasets finales

In [ ]:
# ============================================================
# PASO 24 - GUARDAR DATASET SOILGRIDS PROVINCIAL DEFINITIVO
# ============================================================

import os

columnas_finales_soil = [
    "provincia_estandar",
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm"
]

soil_final = (
    soil_provincial[columnas_finales_soil]
    .copy()
    .sort_values("provincia_estandar")
    .reset_index(drop=True)
)

ruta_soil_final = os.path.join(
    RUTA_PROCESADOS,
    "SoilGrids_provincial_0_30cm.csv"
)

soil_final.to_csv(
    ruta_soil_final,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado:")
print(ruta_soil_final)

print("\nExiste:", os.path.exists(ruta_soil_final))
print("Dimensiones:", soil_final.shape)

display(soil_final.head(10))

In [ ]:
# ============================================================
# PASO 25 - GUARDAR DATOS SOILGRIDS POR PROFUNDIDAD
# ============================================================

ruta_soil_profundidades = os.path.join(
    RUTA_PROCESADOS,
    "SoilGrids_provincial_por_profundidad.csv"
)

soil_long.to_csv(
    ruta_soil_profundidades,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado:")
print(ruta_soil_profundidades)

print("\nExiste:", os.path.exists(ruta_soil_profundidades))
print("Dimensiones:", soil_long.shape)

In [ ]:
# ============================================================
# PASO 26 - VERIFICAR ARCHIVO DEFINITIVO DESDE DRIVE
# ============================================================

import pandas as pd

soil_check = pd.read_csv(
    ruta_soil_final
)

print("Dimensiones:", soil_check.shape)
print("Provincias únicas:", soil_check["provincia_estandar"].nunique())
print("Valores nulos totales:", soil_check.isna().sum().sum())

print("\nColumnas:")
print(soil_check.columns.tolist())

display(soil_check.head())